# This notebook converts the .mat data to netcdf format, 
# add the necessary metadata and plot the data

## load library

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os, sys
import json

In [2]:
# Import modules
%reload_ext autoreload
%autoreload 2

# from src.netcdf import mat_to_xarray
sys.path.append('/Users/yugao/UOP/ORS-processing/src')
from metadata import create_json
from netcdf import read_mat_file, mat_to_xarray, save_to_netcdf

## INPUT REQUIRED: 
## Depth parameters from mooring diagram and mooring log 

In [3]:
instrument_height_above_bottom =  39   
water_depth_from_mooring_diagram = 4450
water_depth_from_ship_uncorrected = 4562.2      # uncorrected water depth 
water_depth_from_ship_corrected = 4538.97       # Replace with actual value, best water depth

#  Anchor (1) + chain (5) + nystron (20) + chain (5) + releases (1) + chain (5) 
# + 6 terminations at 0.25 ea (1.5) 
# + distance from termination on SBE cage to sensor (0.5) = 39 m

In [4]:
depth_parameters = {
    # consult mooring diagram
    'water_depth_from_mooring_diagram': water_depth_from_mooring_diagram,  
     # uncorrected water depth
    'water_depth_from_ship_uncorrected': water_depth_from_ship_uncorrected,
    # Replace with actual value, best water depth
    'water_depth_from_ship_corrected': water_depth_from_ship_corrected,        
    # instrument_depth_from_mooring_diagram = diagram depth  - height above bottom 
    'instrument_depth_from_mooring_diagram': water_depth_from_mooring_diagram - instrument_height_above_bottom,  
    # instrument_depth_from_mooring_log = corrected depth (4538.97 m) - height above bottom (39 m) 
    'instrument_depth_from_mooring_log': water_depth_from_ship_corrected - instrument_height_above_bottom,
    'instrument_height_above_bottom': 39     
    #  Anchor (1) + chain (5) + nystron (20) + chain (5) + releases (1) + chain (5) 
    # + 6 terminations at 0.25 ea (1.5) 
    # + distance from termination on SBE cage to sensor (0.5) = 39 m
}
depth_parameters 

{'water_depth_from_mooring_diagram': 4450,
 'water_depth_from_ship_uncorrected': 4562.2,
 'water_depth_from_ship_corrected': 4538.97,
 'instrument_depth_from_mooring_diagram': 4411,
 'instrument_depth_from_mooring_log': 4499.97,
 'instrument_height_above_bottom': 39}

## read mat file

In [5]:
mat_filepath = '../../data/external/stratus12_sbe16_1876.mat'
mat_data = read_mat_file(mat_filepath)
# Inspect the structure of mat_data
# print(mat_data.keys()) 

## convert mat file to netcdf

In [6]:
sbe16_metadata = mat_data['meta']

In [7]:
# Use the loaded MATLAB data (mat_data_updated) for conversion
ds = mat_to_xarray(mat_data)

## Meta data: we will add depth paramter for deep T/S

In [8]:
metadata_dict = create_json(mat_data, depth_parameters)

json_path = '/Users/yugao/UOP/ORS-processing/data/metadata/' + mat_filepath[-24:-4] + '.json'

# Convert the dictionary to a JSON string and save it
json_path =  mat_filepath[-24:-4] + '.json'
with open(json_path, 'w') as f:
    json.dump(metadata_dict, f)

In [9]:
# meta = mat_data.get('meta', {}) if isinstance(mat_data, dict) else {}
# meta

### read json file

In [10]:
with open(json_path, 'r') as f:
    metadata_from_json = json.load(f)

### incorporate the json file with all attributes

In [11]:
# # Direct keys to extract from 'meta'
# direct_keys = ['site', 'deployment', 'experiment', 'principal_investigator', 'platform', 'global']

# # Process structured array 'meta' to extract direct keys
# try:
#     for name in direct_keys:
#         if name in meta.dtype.names:
#             print('found' + name)
#             # Direct attributes from 'meta'
#             attr_value = meta[name][()]
#             attributes[name] = attr_value if isinstance(attr_value, (int, float, str)) else str(attr_value)
# except Exception as e:
#     print(f"Failed to process 'meta' for direct keys: {e}")


In [12]:
# attributes

In [13]:
metadata_from_json

{'attributes': {'site': 'Stratus',
  'deployment': '12',
  'experiment': 'Stratus Ocean Reference Station',
  'principal_investigator': 'Robert Weller',
  'platform': "('surface mooring', 2012, '27-May-2012 22:03:04', '03-Mar-2014 12:54:00', 735016.9791666666, 735624.5833333334, array([735016.9187963 , 735666.45549769]), 607.6645370370243, 644.6187037036289, 4539, 65, 3.7, 'Melville MV12-07', 'Ron Brown RB14-01', array((6.97132, -6.79155, 'NOAA > NESDIS > NGDC', 'http://www.ngdc.noaa.gov/geomag-web/#declination', 'IGRF11', '27-Mar-2013'),\n      dtype=[('value_to_be_applied', 'O'), ('changing_by_MinutesPerYear', 'O'), ('source', 'O'), ('URL', 'O'), ('model', 'O'), ('midpoint_date', 'O')]))",
  'global': "('WHOI', 'WHOI-UOP', 'Mooring observation', 'OceanSITES', 'Station', '38400')",
  'time_coverage_start': '2012-05-14T00:59:59Z',
  'time_coverage_end': '2014-03-09T00:29:59Z',
  'time_coverage_duration': 'P663D',
  'geospatial_lat_min': -19.93844,
  'geospatial_lat_max': -19.93844,
  '

In [14]:
# Flatten the nested dictionaries within the 'attributes' attribute
attributes_flat = {}
for key, value in  metadata_from_json.items():
    if isinstance(value, dict):
        for sub_key, sub_value in value.items():
            attributes_flat[f'{sub_key}'] = sub_value
    else:
        attributes_flat[key] = value

# Remove the 'attributes' attribute
# if 'attributes' in ds.attrs:
#     del ds.attrs['attributes']

# Update the attributes with the flattened dictionary
ds.attrs.update(attributes_flat)

In [15]:
ds.attrs

{'depth': 4502,
 'latitude': -19.93844,
 'longitude': -85.29266333333334,
 'site': 'Stratus',
 'deployment': '12',
 'experiment': 'Stratus Ocean Reference Station',
 'principal_investigator': 'Robert Weller',
 'platform': "('surface mooring', 2012, '27-May-2012 22:03:04', '03-Mar-2014 12:54:00', 735016.9791666666, 735624.5833333334, array([735016.9187963 , 735666.45549769]), 607.6645370370243, 644.6187037036289, 4539, 65, 3.7, 'Melville MV12-07', 'Ron Brown RB14-01', array((6.97132, -6.79155, 'NOAA > NESDIS > NGDC', 'http://www.ngdc.noaa.gov/geomag-web/#declination', 'IGRF11', '27-Mar-2013'),\n      dtype=[('value_to_be_applied', 'O'), ('changing_by_MinutesPerYear', 'O'), ('source', 'O'), ('URL', 'O'), ('model', 'O'), ('midpoint_date', 'O')]))",
 'global': "('WHOI', 'WHOI-UOP', 'Mooring observation', 'OceanSITES', 'Station', '38400')",
 'time_coverage_start': '2012-05-14T00:59:59Z',
 'time_coverage_end': '2014-03-09T00:29:59Z',
 'time_coverage_duration': 'P663D',
 'geospatial_lat_min':

In [16]:
ds

<xarray.Dataset> Size: 1MB
Dimensions:  (time: 31872)
Coordinates:
  * time     (time) float64 255kB 7.35e+05 7.35e+05 ... 7.357e+05 7.357e+05
Data variables:
    temp     (time) float64 255kB 17.31 17.21 17.1 16.97 ... 18.66 19.51 19.51
    cond     (time) float64 255kB -7.8e-05 -7.8e-05 ... 0.000535 0.000535
    sal      (time) float64 255kB 0.008699 0.008652 0.008598 ... 0.01086 0.01086
    sal_sbe  (time) float64 255kB 0.0 0.0 0.0 0.0 ... 0.0269 0.0109 0.0109
Attributes: (12/25)
    depth:                                  4502
    latitude:                               -19.93844
    longitude:                              -85.29266333333334
    site:                                   Stratus
    deployment:                             12
    experiment:                             Stratus Ocean Reference Station
    ...                                     ...
    water_depth_from_mooring_diagram:       4450
    water_depth_from_ship_uncorrected:      4562.2
    water_depth_from_ship_corrected:        4538.97
    instrument_depth_from_mooring_diagram:  4411
    instrument_depth_from_mooring_log:      4499.97
    instrument_height_above_bottom:         39

In [17]:
netcdf_path = '/Users/yugao/UOP/ORS-processing/data/processed/stratus/' + mat_filepath[-24:-4] + '.nc'

In [18]:
# # Save the dataset to netCDF
ds.to_netcdf(netcdf_path)

In [20]:
ds.attrs

{'depth': 4502,
 'latitude': -19.93844,
 'longitude': -85.29266333333334,
 'site': 'Stratus',
 'deployment': '12',
 'experiment': 'Stratus Ocean Reference Station',
 'principal_investigator': 'Robert Weller',
 'platform': "('surface mooring', 2012, '27-May-2012 22:03:04', '03-Mar-2014 12:54:00', 735016.9791666666, 735624.5833333334, array([735016.9187963 , 735666.45549769]), 607.6645370370243, 644.6187037036289, 4539, 65, 3.7, 'Melville MV12-07', 'Ron Brown RB14-01', array((6.97132, -6.79155, 'NOAA > NESDIS > NGDC', 'http://www.ngdc.noaa.gov/geomag-web/#declination', 'IGRF11', '27-Mar-2013'),\n      dtype=[('value_to_be_applied', 'O'), ('changing_by_MinutesPerYear', 'O'), ('source', 'O'), ('URL', 'O'), ('model', 'O'), ('midpoint_date', 'O')]))",
 'global': "('WHOI', 'WHOI-UOP', 'Mooring observation', 'OceanSITES', 'Station', '38400')",
 'time_coverage_start': '2012-05-14T00:59:59Z',
 'time_coverage_end': '2014-03-09T00:29:59Z',
 'time_coverage_duration': 'P663D',
 'geospatial_lat_min':